In [18]:

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()


Q1 — Dépôt du fichier brut dans SeaweedFS

In [19]:
import os
import boto3
s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:8333",
    aws_access_key_id="admin",
    aws_secret_access_key="password",
    region_name="us-east-1",
)
print("Buckets :", [b["Name"] for b in s3.list_buckets()["Buckets"]])
spark.sql("SHOW NAMESPACES IN lake").show()


Buckets : ['lakehouse', 'landing', 'mlflow']
+---------+
|namespace|
+---------+
|   bronze|
|  default|
|     gold|
|   silver|
+---------+



In [20]:
from pathlib import Path

BUCKET = "lakehouse"
PREFIXE = "landing/afrimarket/commandes"
FICHIER = next(
    p for p in [Path("data"), Path("../data")] if p.is_dir()
) / "afrimarket_dataset_senior.csv"

# 1. Dépôt du fichier brut
s3.upload_file(str(FICHIER), BUCKET, f"{PREFIXE}/{FICHIER.name}")
print(f"déposé : s3://{BUCKET}/{PREFIXE}/{FICHIER.name}")

# 2. Liste des objets du préfixe avec leur taille
for objet in s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIXE}/")["Contents"]:
    print(f"{objet['Key']:60} {objet['Size']:>9} octets")

déposé : s3://lakehouse/landing/afrimarket/commandes/afrimarket_dataset_senior.csv
landing/afrimarket/commandes/afrimarket_dataset_senior.csv     1218028 octets


Q2 — Lecture sans inférence de types

In [21]:


CHEMIN = "s3a://lakehouse/landing/afrimarket/commandes/afrimarket_dataset_senior.csv"

COLONNES = [
    "id_commande", "date_commande", "id_client", "ville", "categorie",
    "nom_produit", "prix_unitaire", "quantite", "remise", "cout_livraison",
    "methode_paiement", "canal_marketing", "cout_marketing", "statut_commande",
]

schema = StructType([StructField(nom, StringType(), True) for nom in COLONNES])

df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(CHEMIN)
)

df.printSchema()
print("Nombre de lignes :", df.count())
df.show(5, truncate=False)

root
 |-- id_commande: string (nullable = true)
 |-- date_commande: string (nullable = true)
 |-- id_client: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- nom_produit: string (nullable = true)
 |-- prix_unitaire: string (nullable = true)
 |-- quantite: string (nullable = true)
 |-- remise: string (nullable = true)
 |-- cout_livraison: string (nullable = true)
 |-- methode_paiement: string (nullable = true)
 |-- canal_marketing: string (nullable = true)
 |-- cout_marketing: string (nullable = true)
 |-- statut_commande: string (nullable = true)

Nombre de lignes : 10100
+-----------+-------------+---------+-----------+------------+-----------------------+-------------+--------+------+--------------+-----------------------+---------------+--------------+---------------+
|id_commande|date_commande|id_client|ville      |categorie   |nom_produit            |prix_unitaire|quantite|remise|cout_livraison|methode_paiement       |cana

Q3, colonnes techniques et table Bronze

ajout des trois colonnes

In [22]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

horodatage = datetime.now()   # calculé une seule fois pour tout le lot

df_bronze = (
    df
    .withColumn("_source_fichier", F.col("_metadata.file_name"))
    .withColumn("_date_ingestion", F.lit(horodatage))
    .withColumn(
        "_num_ligne",
        F.row_number().over(Window.orderBy(F.monotonically_increasing_id())),
    )
)

df_bronze.select("id_commande", "_source_fichier", "_date_ingestion", "_num_ligne").show(5, truncate=False)

+-----------+-----------------------------+--------------------------+----------+
|id_commande|_source_fichier              |_date_ingestion           |_num_ligne|
+-----------+-----------------------------+--------------------------+----------+
|CMD100000  |afrimarket_dataset_senior.csv|2026-11-21 22:12:46.589895|1         |
|CMD100001  |afrimarket_dataset_senior.csv|2026-11-21 22:12:46.589895|2         |
|CMD100002  |afrimarket_dataset_senior.csv|2026-11-21 22:12:46.589895|3         |
|CMD100003  |afrimarket_dataset_senior.csv|2026-11-21 22:12:46.589895|4         |
|CMD100004  |afrimarket_dataset_senior.csv|2026-11-21 22:12:46.589895|5         |
+-----------+-----------------------------+--------------------------+----------+
only showing top 5 rows


écriture dans la table Iceberg

In [23]:
TABLE = "lake.bronze.commandes_raw"

df_bronze.writeTo(TABLE).using("iceberg").createOrReplace()

print("Lignes dans la table :", spark.table(TABLE).count())

Lignes dans la table : 10100


Q4, doublons

Q4a : cellule de la requête

In [24]:
resultat = spark.sql("""
    SELECT
        COUNT(*)     AS ids_en_double,
        SUM(nb - 1)  AS lignes_en_trop
    FROM (
        SELECT id_commande, COUNT(*) AS nb
        FROM lake.bronze.commandes_raw
        GROUP BY id_commande
        HAVING COUNT(*) > 1
    ) t
""")
resultat.show()

+-------------+--------------+
|ids_en_double|lignes_en_trop|
+-------------+--------------+
|          100|           100|
+-------------+--------------+



In [25]:
spark.sql("""
    SELECT nb AS occurrences, COUNT(*) AS nb_identifiants
    FROM (
        SELECT id_commande, COUNT(*) AS nb
        FROM lake.bronze.commandes_raw
        GROUP BY id_commande
    ) t
    GROUP BY nb
    ORDER BY nb
""").show()

+-----------+---------------+
|occurrences|nb_identifiants|
+-----------+---------------+
|          1|           9900|
|          2|            100|
+-----------+---------------+



Q5, villes et statuts

reproduire les deux sorties du sujet

In [26]:
bronze = spark.table("lake.bronze.commandes_raw")

bronze.groupBy("ville").count().orderBy(F.desc("count")).show()
bronze.groupBy("statut_commande").count().show()

+-----------+-----+
|      ville|count|
+-----------+-----+
|   Kinshasa| 2349|
|    Abidjan| 1886|
|     Douala| 1449|
|      Dakar| 1434|
|       Lomé|  735|
|    Cotonou|  677|
|  Kinshassa|  605|
| Libreville|  493|
|Brazzaville|  472|
+-----------+-----+

+---------------+-----+
|statut_commande|count|
+---------------+-----+
|        Annulée|  194|
|      retournée|  826|
|         Livrée| 9080|
+---------------+-----+



code de correction

In [27]:
corrige = (
    bronze
    .withColumn(
        "ville",
        F.when(F.col("ville") == "Kinshassa", "Kinshasa").otherwise(F.col("ville")),
    )
    .withColumn(
        "statut_commande",
        F.initcap(F.lower(F.trim(F.col("statut_commande")))),
    )
)

corrige.groupBy("ville").count().orderBy(F.desc("count")).show()
corrige.groupBy("statut_commande").count().show()

+-----------+-----+
|      ville|count|
+-----------+-----+
|   Kinshasa| 2954|
|    Abidjan| 1886|
|     Douala| 1449|
|      Dakar| 1434|
|       Lomé|  735|
|    Cotonou|  677|
| Libreville|  493|
|Brazzaville|  472|
+-----------+-----+

+---------------+-----+
|statut_commande|count|
+---------------+-----+
|        Annulée|  194|
|      Retournée|  826|
|         Livrée| 9080|
+---------------+-----+



In [28]:
corrige = (bronze
    .withColumn("ville", F.when(F.col("ville") == "Kinshassa", "Kinshasa").otherwise(F.col("ville")))
    .withColumn("statut_commande", F.initcap(F.lower(F.trim(F.col("statut_commande"))))))

Q6, cohérence catégorie / produit

expression d’extraction

In [29]:
cat_extraite = F.split(F.col("nom_produit"), "_").getItem(1)

reproduire le tableau croisé du sujet

In [30]:
corrige = corrige.withColumn("categorie_extraite", cat_extraite)

corrige.groupBy("categorie").pivot("categorie_extraite").count().na.fill(0).show()

+------------+------+------+----+------------+
|   categorie|Beauté|Maison|Mode|Électronique|
+------------+------+------+----+------------+
|electronique|   137|   127| 175|         167|
|Électronique|     0|     0|   0|        2638|
|      Beauté|  1903|     0|   0|           0|
|      Maison|     0|  1948|   0|           0|
|        Mode|     0|     0|3005|           0|
+------------+------+------+----+------------+



comptage des lignes incohérentes

In [31]:
def normaliser(c):
    return F.translate(F.lower(F.trim(c)), "éèêëàâîïôûùç", "eeeeaaiiouuc")

incoherentes = corrige.filter(
    normaliser(F.col("categorie")) != normaliser(F.col("categorie_extraite"))
)
print("Lignes incohérentes :", incoherentes.count())

Lignes incohérentes : 439
